In [13]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from scipy.ndimage import convolve

In [14]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_IMAGES_DIR = PROCESSED_DIR / "processed_images"

GRAY_DIR = PROCESSED_IMAGES_DIR / "gray"
BINARY_DIR = PROCESSED_IMAGES_DIR / "binary"
SKELETON_DIR = PROCESSED_IMAGES_DIR / "skeleton"

image_mapping = pd.read_csv(PROCESSED_DIR / "image_mapping.csv")

print(image_mapping.shape)
image_mapping.head()

(89, 3)


,image_name,image_path,case_id
0,١_page-0001.jpg,d:\FCAI\Year 3\2nd semster\Cognitive\bender-ge...,1
1,١_page-0002.jpg,d:\FCAI\Year 3\2nd semster\Cognitive\bender-ge...,2
2,١_page-0003.jpg,d:\FCAI\Year 3\2nd semster\Cognitive\bender-ge...,3
3,١_page-0004.jpg,d:\FCAI\Year 3\2nd semster\Cognitive\bender-ge...,4
4,١_page-0005.jpg,d:\FCAI\Year 3\2nd semster\Cognitive\bender-ge...,5


In [15]:
def count_skeleton_points(skeleton_img):
    """
    Counts endpoints and intersections in a skeleton image.
    Endpoint = skeleton pixel with 1 neighbor.
    Intersection = skeleton pixel with 3 or more neighbors.
    """
    
    ## skeleton make the drwaing lines to be one pixel wide(very thin) to make it easier to analyze the structure of the
    # drawing and extract features like endpoints and intersections.
    ## this line to make sure that the image is binary and in uint8(unsigned integer) format (0 and 1)
    skel = (skeleton_img > 0).astype(np.uint8)  

    kernel = np.array([ ## the mid pixel is zero bc we wanna only count neighbors not the pixel itself
        [1, 1, 1],
        [1, 0, 1],
        [1, 1, 1]
    ], dtype=np.uint8)

    neighbor_count = convolve(skel, kernel, mode="constant", cval=0) ## cval mean like zero padding around image
## endpoint is a pixel in the skeleton that has only one neighbor, which indicates the end of a line segment.
## intersection is a pixel in the skeleton that has three or more neighbors, which indicates a junction
    endpoints = np.logical_and(skel == 1, neighbor_count == 1).sum()
    intersections = np.logical_and(skel == 1, neighbor_count >= 3).sum()

    return int(endpoints), int(intersections)

In [16]:
def extract_image_features(case_id):
    gray_path = GRAY_DIR / f"case_{case_id:03d}_gray.png"
    binary_path = BINARY_DIR / f"case_{case_id:03d}_binary.png"
    skeleton_path = SKELETON_DIR / f"case_{case_id:03d}_skeleton.png"

    gray = cv2.imread(str(gray_path), cv2.IMREAD_GRAYSCALE)
    binary = cv2.imread(str(binary_path), cv2.IMREAD_GRAYSCALE)
    skeleton = cv2.imread(str(skeleton_path), cv2.IMREAD_GRAYSCALE)

    if gray is None or binary is None or skeleton is None:
        raise ValueError(f"Missing processed image for case {case_id}")

    h, w = binary.shape
    image_area = h * w

# ink pixels indentifies the number of pixels that are part of the drawing 
# what it can tell us? -> more drawing, more details, thicker lines ( but its ageneral feature )
    ink_pixels = int(np.count_nonzero(binary))
    ink_density = ink_pixels / image_area

# tryin to measure the lenght of the drawing lines without being affected by thikness
# what it can tell us? -> more complex drawing with more details will have longer skeleton
    skeleton_pixels = int(np.count_nonzero(skeleton))
    skeleton_density = skeleton_pixels / image_area

# stroke thickness of lines
# what it can tell us? -> thicker lines maybe determin more connfidence or emphasis or more redrawing
    stroke_thickness_proxy = ink_pixels / (skeleton_pixels + 1e-6)


    stroke_mask = binary > 0 #identify the pixels that are part of the drawing (ink pixels)

    if ink_pixels > 0:
        stroke_gray_values = gray[stroke_mask] # extract the gray values only from the pixels that are part of the drawing

        # in gray(0 = black, 255 = white) so we will get from it the values of darkness
        # pressure of the pen or redrawing
        stroke_darkness_values = 255 - stroke_gray_values

        stroke_darkness_mean = float(np.mean(stroke_darkness_values))
        stroke_darkness_std = float(np.std(stroke_darkness_values)) ## to check if the drawing pressure is consistent or not
    else:
        stroke_darkness_mean = 0.0
        stroke_darkness_std = 0.0

    # connected components
    # connected components are groups of connected pixels in the binary image that represent distinct parts of the drawing.
    # num_labels = number of connected components (including background)
    # labels = an image where each pixel is labeled with the component it belongs to
    # stats = statistics for each component (area, bounding box, position , x , y)
    
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary,
        connectivity=8
    )

    component_areas = stats[1:, cv2.CC_STAT_AREA]

    valid_components = component_areas[component_areas > 20]
    
    # small components may include noise or small deatils or Cuts in the line.
    small_components = component_areas[(component_areas > 0) & (component_areas <= 20)]

    connected_components_count = int(len(valid_components))
    small_components_count = int(len(small_components))

    if len(valid_components) > 0:
        largest_component_area = int(valid_components.max())
        mean_component_area = float(valid_components.mean())
        largest_component_area_ratio = largest_component_area / image_area
    else:
        largest_component_area = 0
        mean_component_area = 0.0
        largest_component_area_ratio = 0.0

    # contours
    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    valid_contours = [
        cnt for cnt in contours
        if cv2.contourArea(cnt) > 20
    ]

    contour_count = len(valid_contours)


    # skeleton endpoints and intersections
    # the higher number of endpoints indicates more lines, separate segments, uncomplete connection of drawing
    # the higher number of intersections indicates more complex drawing with more connections and details
    endpoints_count, intersections_count = count_skeleton_points(skeleton)

    # we normalize them by skeleton lenght
    endpoints_per_skeleton_length = endpoints_count / (skeleton_pixels + 1e-6)
    intersections_per_skeleton_length = intersections_count / (skeleton_pixels + 1e-6)
    
    # if the ink area is large and number of components is small means that the drawing is more connected
    # if the ink is small and number of components is large means that the drawing is more fragmented
    components_per_ink_area = connected_components_count / (ink_pixels + 1e-6)

    # bounding box around all drawing pixels
    ys, xs = np.where(binary > 0)

    if len(xs) > 0 and len(ys) > 0:
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()

        bbox_width = int(x_max - x_min + 1)
        bbox_height = int(y_max - y_min + 1)
        bbox_area = bbox_width * bbox_height # indicates the overall size of the drawing and how much space it occupies on the page
        bbox_area_ratio = bbox_area / image_area

        bbox_aspect_ratio = bbox_width / (bbox_height + 1e-6)

        bbox_center_x = (x_min + x_max) / 2
        bbox_center_y = (y_min + y_max) / 2

        bbox_center_x_norm = bbox_center_x / w
        bbox_center_y_norm = bbox_center_y / h

        page_center_x = w / 2
        page_center_y = h / 2

        drawing_center_distance = np.sqrt(
            (bbox_center_x - page_center_x) ** 2 +
            (bbox_center_y - page_center_y) ** 2
        )

        drawing_center_distance_norm = drawing_center_distance / np.sqrt(w**2 + h**2)
    else:
        bbox_width = 0
        bbox_height = 0
        bbox_area_ratio = 0.0
        bbox_aspect_ratio = 0.0
        bbox_center_x_norm = 0.0
        bbox_center_y_norm = 0.0
        drawing_center_distance_norm = 0.0

    # orientation without rotating
    is_landscape = int(w > h)

    features = {
        "case_id": case_id,

        # image size / orientation
        "image_width": w,
        "image_height": h,
        "is_landscape": is_landscape,

        # ink / binary features
        "ink_pixels": ink_pixels,
        "ink_density": ink_density,

        # grayscale pressure features
        "stroke_darkness_mean": stroke_darkness_mean,
        "stroke_darkness_std": stroke_darkness_std,

        # skeleton features
        "skeleton_length": skeleton_pixels,
        "skeleton_density": skeleton_density,
        "stroke_thickness_proxy": stroke_thickness_proxy,

        # components / contours
        "connected_components_count": connected_components_count,
        "small_components_count": small_components_count,
        "largest_component_area": largest_component_area,
        "largest_component_area_ratio": largest_component_area_ratio,
        "mean_component_area": mean_component_area,
        "contour_count": contour_count,

        # skeleton structure
        "endpoints_count": endpoints_count,
        "intersections_count": intersections_count,
        "endpoints_per_skeleton_length": endpoints_per_skeleton_length,
        "intersections_per_skeleton_length": intersections_per_skeleton_length,
        "components_per_ink_area": components_per_ink_area,

        # spatial / bounding box features
        "bbox_width": bbox_width,
        "bbox_height": bbox_height,
        "bbox_area_ratio": bbox_area_ratio,
        "bbox_aspect_ratio": bbox_aspect_ratio,
        "bbox_center_x_norm": bbox_center_x_norm,
        "bbox_center_y_norm": bbox_center_y_norm,
        "drawing_center_distance_norm": drawing_center_distance_norm
    }

    return features

In [17]:
features = extract_image_features(case_id=1)

features

{'case_id': 1,
 'image_width': 1241,
 'image_height': 1755,
 'is_landscape': 0,
 'ink_pixels': 27946,
 'ink_density': 0.012831302758780598,
 'stroke_darkness_mean': 63.72779646461032,
 'stroke_darkness_std': 49.88203516047379,
 'skeleton_length': 7015,
 'skeleton_density': 0.0032209113595092643,
 'stroke_thickness_proxy': 3.983749108484141,
 'connected_components_count': 132,
 'small_components_count': 108,
 'largest_component_area': 4176,
 'largest_component_area_ratio': 0.0019173949874997417,
 'mean_component_area': 206.1060606060606,
 'contour_count': 106,
 'endpoints_count': 376,
 'intersections_count': 59,
 'endpoints_per_skeleton_length': 0.05359942978565938,
 'intersections_per_skeleton_length': 0.008410548822749744,
 'components_per_ink_area': 0.004723395118989358,
 'bbox_width': 1200,
 'bbox_height': 1687,
 'bbox_area_ratio': 0.9294957884804782,
 'bbox_aspect_ratio': 0.7113218727259503,
 'bbox_center_x_norm': np.float64(0.4935535858178888),
 'bbox_center_y_norm': np.float64(0.

In [18]:
all_features = []

for _, row in image_mapping.iterrows():
    case_id = int(row["case_id"])
    features = extract_image_features(case_id)
    all_features.append(features)

image_features = pd.DataFrame(all_features)

print(image_features.shape)
image_features.head()

(89, 29)


,case_id,image_width,image_height,is_landscape,ink_pixels,ink_density,stroke_darkness_mean,stroke_darkness_std,skeleton_length,skeleton_density,...,endpoints_per_skeleton_length,intersections_per_skeleton_length,components_per_ink_area,bbox_width,bbox_height,bbox_area_ratio,bbox_aspect_ratio,bbox_center_x_norm,bbox_center_y_norm,drawing_center_distance_norm
0,1,1241,1755,0,27946,0.012831,63.727796,49.882035,7015,0.003221,...,0.053599,0.008411,0.004723,1200,1687,0.929496,0.711322,0.493554,0.511681,0.010238
1,2,1241,1755,0,33761,0.015501,117.697965,63.608772,6718,0.003085,...,0.063709,0.053885,0.004206,1183,1643,0.892428,0.720024,0.500403,0.478632,0.017448
2,3,1241,1755,0,29999,0.013774,81.395313,53.612199,7692,0.003532,...,0.054082,0.031851,0.004067,1165,1494,0.799149,0.779786,0.478646,0.433903,0.055358
3,4,1755,1241,1,31808,0.014605,94.960670,47.822996,7157,0.003286,...,0.067906,0.025430,0.004464,1512,1155,0.801835,1.309091,0.511396,0.489122,0.011226
4,5,1241,1755,0,33813,0.015525,125.515246,64.123418,7413,0.003404,...,0.054094,0.021179,0.004081,1191,1571,0.859091,0.758116,0.497180,0.479202,0.017059


In [19]:
def safe_zscore(series):
    std = series.std()

    if std == 0 or np.isnan(std):
        return series * 0

    return (series - series.mean()) / std

In [20]:
# Perseveration proxy:
# high stroke thickness + high ink density + high darkness may indicate repeated/heavy drawing
image_features["perseveration_score"] = (
    safe_zscore(image_features["stroke_thickness_proxy"]) +
    safe_zscore(image_features["ink_density"]) +
    safe_zscore(image_features["stroke_darkness_mean"])
) / 3


# Distortion proxy:
# high endpoints/components/intersections may indicate fragmented or structurally distorted drawings
image_features["distortion_proxy_score"] = (
    safe_zscore(image_features["endpoints_per_skeleton_length"]) +
    safe_zscore(image_features["intersections_per_skeleton_length"]) +
    safe_zscore(image_features["connected_components_count"]) +
    safe_zscore(image_features["bbox_area_ratio"])
) / 4

In [21]:
output_path = PROCESSED_DIR / "image_features.csv"

image_features.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Saved to:", output_path)
print("Final image features shape:", image_features.shape)

image_features.head()

Saved to: d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\data\processed\image_features.csv
Final image features shape: (89, 31)


,case_id,image_width,image_height,is_landscape,ink_pixels,ink_density,stroke_darkness_mean,stroke_darkness_std,skeleton_length,skeleton_density,...,components_per_ink_area,bbox_width,bbox_height,bbox_area_ratio,bbox_aspect_ratio,bbox_center_x_norm,bbox_center_y_norm,drawing_center_distance_norm,perseveration_score,distortion_proxy_score
0,1,1241,1755,0,27946,0.012831,63.727796,49.882035,7015,0.003221,...,0.004723,1200,1687,0.929496,0.711322,0.493554,0.511681,0.010238,-1.366521,-0.303671
1,2,1241,1755,0,33761,0.015501,117.697965,63.608772,6718,0.003085,...,0.004206,1183,1643,0.892428,0.720024,0.500403,0.478632,0.017448,0.074930,0.471576
2,3,1241,1755,0,29999,0.013774,81.395313,53.612199,7692,0.003532,...,0.004067,1165,1494,0.799149,0.779786,0.478646,0.433903,0.055358,-1.116456,-0.379011
3,4,1755,1241,1,31808,0.014605,94.960670,47.822996,7157,0.003286,...,0.004464,1512,1155,0.801835,1.309091,0.511396,0.489122,0.011226,-0.597039,-0.231889
4,5,1241,1755,0,33813,0.015525,125.515246,64.123418,7413,0.003404,...,0.004081,1191,1571,0.859091,0.758116,0.497180,0.479202,0.017059,-0.059099,-0.291300


In [22]:
output_path = PROCESSED_DIR / "image_features.csv"

image_features.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Saved to:", output_path)
print("Final image features shape:", image_features.shape)

Saved to: d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\data\processed\image_features.csv
Final image features shape: (89, 31)
